# 使用 GPT-5 进行前端开发

GPT-5 在前端开发方面实现了巨大飞跃。我们看到该模型非常擅长一次性开发出全栈应用，轻松完成复杂的重构，并在大型代码库中进行精准修改。 

在本 cookbook 中，我们将从多个维度展示使用 GPT-5 开发前端应用的一些示例和经验。 

## 简介
我们发现有一些通用原则对开发出色的前端应用很有效。我们在[提示指南](https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide)中分享了其中一些经验。下面是构建前端应用时需要考虑的一些要点。

以下是我们推荐用来引导模型的入门库和包：
- 框架：Next.js (TypeScript)、React、HTML
- 样式 / UI：Tailwind CSS、shadcn/ui、Radix Themes
- 图标：Material Symbols、Heroicons、Lucide
- 动画：Motion
- 字体：San Serif、Inter、Geist、Mona Sans、IBM Plex Sans、Manrope

这些包并非详尽列表，我们也见过许多不同风格的应用。 

下面你会看到一种通过 API 迭代前端抽象的简便方法。我们很期待看到用户如何借助 GPT-5 释放创造力。


## 交互式示例

我们来深入看一个从零开始创建前端的示例。首先，创建一些辅助函数，用来查看 GPT-5 生成的网站。

In [ ]:
import os
import re
import webbrowser
from pathlib import Path

import openai
from openai.types.responses import ResponseInputParam

client = openai.OpenAI()


def get_response_output_text(input: str | ResponseInputParam):
    
    response = client.responses.create(
        model="gpt-5",
        input=input,
    )
    return response.output_text


def extract_html_from_text(text: str):
    """Extract an HTML code block from text; fallback to first code block, else full text."""
    html_block = re.search(r"```html\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if html_block:
        result = html_block.group(1)
        return result
    any_block = re.search(r"```\s*(.*?)\s*```", text, re.DOTALL)
    if any_block:
        result = any_block.group(1)
        return result
    return text


def save_html(html: str, filename: str) -> Path:
    """Save HTML to outputs/ directory and return the path."""
    try:
        base_dir = Path(__file__).parent
    except NameError:
        base_dir = Path.cwd()

    folder = "outputs"
    outputs_dir = base_dir / folder
    outputs_dir.mkdir(parents=True, exist_ok=True)

    output_path = outputs_dir / filename
    output_path.write_text(html, encoding="utf-8")
    return output_path

def open_in_browser(path: Path) -> None:
    """Open a file in the default browser (macOS compatible)."""
    try:
        webbrowser.open(path.as_uri())
    except Exception:
        os.system(f'open "{path}"')



现在，我们把上面的内容合并成一个辅助函数。

In [7]:
def make_website_and_open_in_browser(*, website_input: str | ResponseInputParam, filename: str = "website.html"):
    response_text = get_response_output_text(website_input)
    html = extract_html_from_text(response_text)
    output_path = save_html(html, filename)
    open_in_browser(output_path)

我们从一个简单的示例开始：一次性构建一个主题契合的复古游戏商店

In [8]:
make_website_and_open_in_browser(
    website_input="Make me landing page for a retro-games store. Retro-arcade noir some might say",
    filename="retro_dark.html",
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/retro_dark.html
open_in_browser: finished


对于一行、一次性的提示来说，效果相当不错！

<img src="../../images/retro_dark.png" style="width:60vw; display:block; margin:auto;">


现在我们引导它变得更明亮、更柔和一些

In [9]:
make_website_and_open_in_browser(
    website_input="Make me landing page for a retro-games store. Make it light, more pastel coloured & flowery (think Mario, not cyberpunk)", 
    filename="retro_light.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/retro_light.html
open_in_browser: finished


如你所见，GPT-5 的可引导性非常强——只需一行提示，就能毫不费力地改变整个应用

<img src="../../images/retro_light.png" style="width:60vw; display:block; margin:auto;">

但如果你已有网站设计，想在其基础上添加内容呢？例如，我们已经有了这个仪表盘。

<img src="../../images/input_image.png" style="width:60vw; display:block; margin:auto;">

由于 GPT-5 原生支持多模态，可同时接受图像和文本输入，因此在生成前端应用时，我们可以利用图像输入来提升模型表现。 

In [10]:
import base64
from openai.types.responses import ResponseInputImageParam

# Function to encode the image
def encode_image(image_path: str):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

image_path="../../images/input_image.png"
encoded_image = encode_image(image_path)
input_image: ResponseInputImageParam = {"type": "input_image", "image_url": f"data:image/png;base64,{encoded_image}", "detail": "auto"}
input: ResponseInputParam = [
    {
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Can you make a login page for this website that maintains the same theme"},
            input_image,
        ],
    }
]

make_website_and_open_in_browser(
    website_input=input, 
    filename="login_page.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/login_page.html
open_in_browser: finished


如你所见，GPT-5 出色地匹配了应用现有的风格和氛围。

<img src="../../images/login_page.png" style="width:60vw; display:block; margin:auto;">

到目前为止，这些都相当静态——我们来试一个交互性更强的任务

In [11]:
make_website_and_open_in_browser(
    website_input="Make me a snake game. It should be futuristic, neon, cyberpunk style. Make sure the typography is suitably cool.", 
    filename="snake_game.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/snake_game.html
open_in_browser: finished


我们得到了一个主题一致的贪吃蛇游戏：配色、字体排版甚至音效都相互匹配

<img src="../../images/snake_game.png" style="width:60vw; display:block; margin:auto;">

一条描述不充分的提示也能在一次 API 调用中生成前端代码，但用于生产环境时仍需要审查和迭代。

现在轮到你了——我们迫不及待想看到你会构建出什么

In [12]:
your_prompt = "[edit this! what website would you like to build?]"

make_website_and_open_in_browser(
    website_input=your_prompt, 
    filename="your_website.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/your_website.html
open_in_browser: finished
